# Wikimedia Analytics Pipeline


## Stream Processing Pipeline with Wikimedia Recent Changes Data

Este notebook apresenta a construção de um pipeline de **stream processing** utilizando o stream público de mudanças recentes da Wikimedia e foi desenvolvido e executado na plataforma Databricks.

A fonte utilizada é:

```text
https://stream.wikimedia.org/v2/stream/recentchange
```

O pipeline realiza ingestão de eventos contínuos, armazenamento dos dados brutos em JSON, processamento com Apache Spark Structured Streaming, validação, filtros, conversão para Parquet, agregações analíticas.

A execução foi estruturada de forma finita para fins acadêmicos, utilizando `trigger(availableNow=True)`. Ainda assim, o processamento é realizado com Spark Structured Streaming, por meio de `readStream`, `writeStream`, checkpoints e leitura incremental dos arquivos disponíveis.

## 1. Visão geral da solução


A solução utiliza o Databricks como plataforma de processamento, Apache Spark Structured Streaming como mecanismo de stream processing e Unity Catalog Volume como camada de armazenamento governada.

O objetivo é transformar eventos contínuos da Wikimedia em dados analíticos organizados em camadas:

```text
Wikimedia Recent Changes Stream
        ↓
Raw JSON Events
        ↓
Bronze Parquet
        ↓
Silver Validated Parquet
        ↓
Gold Aggregations
```

Essa estrutura segue uma abordagem inspirada na arquitetura Medallion, favorecendo rastreabilidade, separação lógica das etapas e evolução para um pipeline operacional.

## 2.Setup recursos

### 2.1 Criação do schema e do Unity Catalog Volume

A solução utiliza um Unity Catalog Volume como camada de armazenamento governada para suportar a organização dos dados em Raw, Bronze, Silver e Gold, além dos checkpoints do processamento streaming.

In [0]:
CATALOG_NAME = "workspace"
SCHEMA_NAME = "stream_processing_pipelines"
VOLUME_NAME = "wikimedia_stream_pipeline"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}
COMMENT 'Schema used for stream processing pipeline projects'
""")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}
COMMENT 'Volume used for the WikiStream Analytics Pipeline project'
""")

print(f"Schema configured: {CATALOG_NAME}.{SCHEMA_NAME}")
print(f"Volume configured: {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")

Schema configured: workspace.stream_processing_pipelines
Volume configured: workspace.stream_processing_pipelines.wikimedia_stream_pipeline


In [0]:
# Exibe os metadados do volume criado no Unity Catalog (validação para confirmar que o volume existe e está disponível para uso no pipeline).
spark.sql(f"DESCRIBE VOLUME {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}").show(truncate=False)

+-------------------------+---------+---------------------------+----------------------------+----------------+-----------+---------------------------------------------------------+--------------+-----------------+
|name                     |catalog  |database                   |owner                       |storage_location|volume_type|comment                                                  |securable_type|securable_kind   |
+-------------------------+---------+---------------------------+----------------------------+----------------+-----------+---------------------------------------------------------+--------------+-----------------+
|wikimedia_stream_pipeline|workspace|stream_processing_pipelines|viviane.correa.dev@gmail.com|                |MANAGED    |Volume used for the WikiStream Analytics Pipeline project|VOLUME        |VOLUME_DB_STORAGE|
+-------------------------+---------+---------------------------+----------------------------+----------------+-----------+-----------------

### 2.2 Configuração dos caminhos do pipeline


Os caminhos abaixo organizam os arquivos do pipeline em áreas específicas para ingestão, processamento, outputs analíticos e checkpoints.

Os checkpoints são fundamentais em pipelines de Structured Streaming, pois registram o progresso do processamento incremental e permitem maior controle sobre reprocessamentos.

In [0]:
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

RAW_PATH = f"{VOLUME_PATH}/raw/events"

BRONZE_OUTPUT_PATH = f"{VOLUME_PATH}/bronze/events_parquet"
SILVER_OUTPUT_PATH = f"{VOLUME_PATH}/silver/validated_events_parquet"
GOLD_AGG_OUTPUT_PATH = f"{VOLUME_PATH}/gold/aggregations_parquet"

# Caminhos de checkpoint do Spark Structured Streaming (cada etapa streaming possui seu próprio checkpoint para controlar progresso e evitar reprocessamento indevido).
CHECKPOINT_BRONZE_PATH = f"{VOLUME_PATH}/checkpoints/bronze"
CHECKPOINT_SILVER_PATH = f"{VOLUME_PATH}/checkpoints/silver"
CHECKPOINT_GOLD_IMPACT_PATH = f"{VOLUME_PATH}/checkpoints/gold_impact"
CHECKPOINT_GOLD_USER_PATH = f"{VOLUME_PATH}/checkpoints/gold_user"
CHECKPOINT_GOLD_EVENT_PATH = f"{VOLUME_PATH}/checkpoints/gold_event"

print("Volume path:", VOLUME_PATH)
print("Raw path:", RAW_PATH)
print("Bronze output:", BRONZE_OUTPUT_PATH)
print("Silver output:", SILVER_OUTPUT_PATH)
print("Gold aggregations output:", GOLD_AGG_OUTPUT_PATH)


Volume path: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline
Raw path: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events
Bronze output: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet
Silver output: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet
Gold aggregations output: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet


### 2.3 Feature de Limpeza


O notebook inclui uma opção de limpeza controlada dos diretórios de dados e checkpoints antes da execução.

Essa configuração permite executar a demonstração de forma isolada, evitando que resultados de testes anteriores sejam misturados aos resultados da rodada atual.

Quando a opção `RESET_PIPELINE_OUTPUTS` está ativada, o pipeline é reiniciado para gerar uma evidência limpa da execução.

In [0]:
# Controla se os outputs anteriores devem ser removidos antes de uma nova execução.
# Quando True, a execução começa limpa, sem misturar resultados de testes anteriores.
# Quando False, os dados e checkpoints são preservados para simular continuidade incremental entre execuções.

RESET_PIPELINE_OUTPUTS = True

paths_to_clean = [
    RAW_PATH,
    BRONZE_OUTPUT_PATH,
    SILVER_OUTPUT_PATH,
    GOLD_AGG_OUTPUT_PATH,
    CHECKPOINT_BRONZE_PATH,
    CHECKPOINT_SILVER_PATH,
    CHECKPOINT_GOLD_IMPACT_PATH,
    CHECKPOINT_GOLD_USER_PATH,
    CHECKPOINT_GOLD_EVENT_PATH
]

if RESET_PIPELINE_OUTPUTS:
    for path in paths_to_clean:
        try:
            dbutils.fs.rm(path, recurse=True)
            print(f"Removido: {path}")
        except Exception:
            print(f"Caminho não encontrado ou não removido: {path}")

    print("Outputs e checkpoints anteriores foram removidos.")
else:
    print("Outputs e checkpoints anteriores foram preservados.")

Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/bronze
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/silver
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/gold_impact
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/gold_user
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/gold_event
Outputs e checkpoints anteriores foram removidos.


In [0]:
paths = [
    RAW_PATH,
    BRONZE_OUTPUT_PATH,
    SILVER_OUTPUT_PATH,
    GOLD_AGG_OUTPUT_PATH,
    CHECKPOINT_BRONZE_PATH,
    CHECKPOINT_SILVER_PATH,
    CHECKPOINT_GOLD_IMPACT_PATH,
    CHECKPOINT_GOLD_USER_PATH,
    CHECKPOINT_GOLD_EVENT_PATH
]

for path in paths:
    dbutils.fs.mkdirs(path)

print("Diretórios do pipeline configurados com sucesso.")
print("Volume path:", VOLUME_PATH)

Diretórios do pipeline configurados com sucesso.
Volume path: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline


In [0]:
# Evidência de que os diretórios foram criados corretamente
display(dbutils.fs.ls(VOLUME_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/,bronze/,0,1777846004356
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/,checkpoints/,0,1777846004356
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/,gold/,0,1777846004356
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/,raw/,0,1777846004356
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/,silver/,0,1777846004356


## 3. Ingestão: Wikimedia Recent Changes

A fonte escolhida é o stream público de mudanças recentes da Wikimedia, que publica eventos continuamente à medida que alterações ocorrem em projetos como Wikipedia, Wikidata, Wikimedia Commons e outros.

O endpoint disponibiliza eventos em tempo real utilizando o padrão **Server-Sent Events (SSE)**.

Neste notebook, a conexão SSE é consumida com a biblioteca `requests`, mantendo a conexão HTTP aberta com `stream=True` e processando as linhas recebidas por meio de `response.iter_lines()`.

As mensagens de evento são identificadas pelo prefixo `data:`, convertidas de JSON para dicionários Python e armazenadas na área Raw em formato JSON Lines.

In [0]:
import json
import time
import requests
from datetime import datetime, timezone

STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"


def ingest_wikimedia_stream(duration_seconds=60, max_events=300):
    """
    Consome o stream Wikimedia Recent Changes e grava os eventos em JSON Lines
    na camada Raw do Unity Catalog Volume.

    Parâmetros:
    duration_seconds: tempo máximo de ingestão em segundos.
    max_events: quantidade máxima de eventos a serem capturados.
    """

    start_time = time.time()
    event_count = 0

    output_file = (
        f"{RAW_PATH}/wikimedia_events_"
        f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json"
    )

    headers = {
        "User-Agent": "WikiStreamAnalyticsPipeline/1.0"
    }

    # O parâmetro stream=True mantém a conexão aberta para leitura incremental das mensagens.
    with requests.get(STREAM_URL, headers=headers, stream=True, timeout=120) as response:

        response.raise_for_status()

        with open(output_file, "w", encoding="utf-8") as file:

            for line in response.iter_lines(decode_unicode=True):

                if time.time() - start_time > duration_seconds:
                    break

                if event_count >= max_events:
                    break

                if not line or not line.startswith("data: "):
                    continue

                payload = line.replace("data: ", "", 1)

                try:
                    data = json.loads(payload)
                    data["_ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
                    file.write(json.dumps(data, ensure_ascii=False) + "\n")
                    event_count += 1

                except json.JSONDecodeError:
                    continue

    print("Ingestão concluída.")
    print(f"Eventos ingeridos: {event_count}")
    print(f"Arquivo gerado: {output_file}")

In [0]:
ingest_wikimedia_stream(duration_seconds=360, max_events=10000)

Ingestão concluída.
Eventos ingeridos: 10000
Arquivo gerado: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events/wikimedia_events_20260503_220645.json


In [0]:
display(dbutils.fs.ls(RAW_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events/wikimedia_events_20260503_220645.json,wikimedia_events_20260503_220645.json,14419578,1777846073000


### 3.1 Definição do schema dos eventos

Para processar os eventos com Spark Structured Streaming, o schema dos principais campos é declarado explicitamente.

Essa prática evita inferência automática em dados contínuos, melhora a estabilidade do pipeline e facilita a aplicação de validações e transformações posteriores.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    LongType,
    IntegerType
)

wikimedia_schema = StructType([
    StructField("$schema", StringType(), True),
    StructField("meta", StructType([
        StructField("uri", StringType(), True),
        StructField("request_id", StringType(), True),
        StructField("id", StringType(), True),
        StructField("dt", StringType(), True),
        StructField("domain", StringType(), True),
        StructField("stream", StringType(), True),
        StructField("topic", StringType(), True),
        StructField("partition", IntegerType(), True),
        StructField("offset", LongType(), True)
    ]), True),
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("title_url", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("patrolled", BooleanType(), True),
    StructField("length", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True)
    ]), True),
    StructField("revision", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True)
    ]), True),
    StructField("server_url", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("server_script_path", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("parsedcomment", StringType(), True),
    StructField("_ingestion_timestamp", StringType(), True)
])

## 4. Camada Bronze


A camada Bronze representa a primeira persistência dos eventos brutos após a ingestão.

Nesta etapa, o Spark lê os arquivos JSON da área Raw utilizando `readStream`, processa os arquivos de forma incremental e grava a saída em Parquet utilizando `writeStream`.

In [0]:
raw_stream_df = (
    spark.readStream
    .schema(wikimedia_schema)
    .option("maxFilesPerTrigger", 10)
    .json(RAW_PATH)
)
print("Is raw_stream_df streaming?", raw_stream_df.isStreaming)

raw_stream_df.printSchema()

Is raw_stream_df streaming? True
root
 |-- $schema: string (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- uri: string (nullable = true)
 |    |-- request_id: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- dt: string (nullable = true)
 |    |-- domain: string (nullable = true)
 |    |-- stream: string (nullable = true)
 |    |-- topic: string (nullable = true)
 |    |-- partition: integer (nullable = true)
 |    |-- offset: long (nullable = true)
 |-- id: long (nullable = true)
 |-- type: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- title_url: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- user: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- patrolled: boolean (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: long (nullable = true)
 |    |-- ne

In [0]:
bronze_query = (
    raw_stream_df.writeStream
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_BRONZE_PATH)
    .option("path", BRONZE_OUTPUT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

bronze_query.awaitTermination()

print("Bronze streaming query completed.")
print("Query status:", bronze_query.status)
print("Last progress:", bronze_query.lastProgress)

Bronze streaming query completed.
Query status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Last progress: {'id': '2aa4bb73-9bba-400c-a668-8eabeb06d329', 'runId': '78f77fee-811c-4efb-b156-a1358fddf7da', 'name': None, 'timestamp': '2026-05-03T22:09:47.847Z', 'batchId': 0, 'batchDuration': 1192, 'durationMs': {'triggerExecution': 1192, 'queryPlanning': 32, 'collectSourceMetrics': 0, 'getBatch': 180, 'commitOffsets': 78, 'addBatch': 733, 'latestOffset': 73, 'walCommit': 184}, 'eventTime': {}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events]', 'startOffset': None, 'endOffset': '{"logOffset":0}', 'latestOffset': None, 'numInputRows': 10000, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 8389.261744966443, 'metrics': {}}], 'sink': {'description': 'FileSink[/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parque

In [0]:
bronze_df = spark.read.parquet(BRONZE_OUTPUT_PATH)

display(bronze_df.limit(20))

print(f"Total records in Bronze: {bronze_df.count()}")

$schema,meta,id,type,namespace,title,title_url,comment,timestamp,user,bot,minor,patrolled,length,revision,server_url,server_name,server_script_path,wiki,parsedcomment,_ingestion_timestamp
/mediawiki/recentchange/1.0.0,"List(https://commons.wikimedia.org/wiki/Category:Books_in_the_National_Library_in_Warsaw, 43156a47-63ab-4e81-97b7-41023167b605, 70a2a7ae-b094-4f93-8fbf-88edf44edb9c, 2026-05-03T22:07:33.869Z, commons.wikimedia.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6094953402)",3305921420,categorize,14,Category:Books in the National Library in Warsaw,https://commons.wikimedia.org/wiki/Category:Books_in_the_National_Library_in_Warsaw,[[:Category:O hodowli drzew i krzewów owocowych w ogródkach małych gospodarstw (1908)]] added to category,1777846052,Redaktor GLAM,false,null,null,null,null,https://commons.wikimedia.org,commons.wikimedia.org,/w,commonswiki,Category:O hodowli drzew i krzewów owocowych w ogródkach małych gospodarstw (1908) added to category,2026-05-03T22:07:33.947533+00:00
/mediawiki/recentchange/1.0.0,"List(https://www.wikidata.org/wiki/Q8003637, 6a3a65ff-ec75-4be7-92c9-c4af961f4baf, ff4779e8-0e04-40b7-af83-9d553424f6e4, 2026-05-03T22:07:33.911Z, www.wikidata.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6094953403)",2564408754,edit,0,Q8003637,https://www.wikidata.org/wiki/Q8003637,/* wbeditentity-update:0| */ Added [[wikipedia:ur:ولسن، او کلیئر کاؤٹی، وسکونسن]],1777846053,KaleemBot,true,false,true,"List(27261, 27397)","List(2455532461, 2487040769)",https://www.wikidata.org,www.wikidata.org,/w,wikidatawiki,‎Updated Item: Added wikipedia:ur:ولسن، او کلیئر کاؤٹی، وسکونسن,2026-05-03T22:07:33.967787+00:00
/mediawiki/recentchange/1.0.0,"List(https://ce.wikipedia.org/wiki/%D0%9A%D0%B0%D1%82%D0%B5%D0%B3%D0%BE%D1%80%D0%B8:WikiProject_Geography_articles, ba996926-a940-436e-966a-2ecabc7bfe0c, f62211b2-137a-4ce2-aeb1-17fca43723ff, 2026-05-03T22:07:33.906Z, ce.wikipedia.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6094953404)",80303672,categorize,14,Категори:WikiProject Geography articles,https://ce.wikipedia.org/wiki/%D0%9A%D0%B0%D1%82%D0%B5%D0%B3%D0%BE%D1%80%D0%B8:WikiProject_Geography_articles,[[:Дийцар:Серпово (Печорин кӀошт)]] категори чу тоьхна,1777846052,CheWikibot,true,null,null,null,null,https://ce.wikipedia.org,ce.wikipedia.org,/w,cewiki,Дийцар:Серпово (Печорин кӀошт) категори чу тоьхна,2026-05-03T22:07:33.969597+00:00
/mediawiki/recentchange/1.0.0,"List(https://commons.wikimedia.org/wiki/Category:Media_lacking_author_information, 50b3fea4-0ae8-49d2-aa7c-12d95467f2db, f3eda0eb-1e63-47f0-ba19-3477b851737c, 2026-05-03T22:07:34.046Z, commons.wikimedia.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6094953405)",3305921417,categorize,14,Category:Media lacking author information,https://commons.wikimedia.org/wiki/Category:Media_lacking_author_information,[[:File:Seating overlooking white water course - geograph.org.uk - 8284980.jpg]] added to category,1777846052,GeographBot,true,null,null,null,null,https://commons.wikimedia.org,commons.wikimedia.org,/w,commonswiki,File:Seating overlooking white water course - geograph.org.uk - 8284980.jpg added to category,2026-05-03T22:07:34.106002+00:00
/mediawiki/recentchange/1.0.0,"List(https://tr.wiktionary.org/wiki/Bulin, c31309b0-0d91-4bae-a208-fbb6d737efc6, d959c580-3a59-439f-9e6f-2b14aa390448, 2026-05-03T22:07:34.072Z, tr.wiktionary.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6094953406)",11516448,new,0,Bulin,https://tr.wiktionary.org/wiki/Bulin,yeni sayfa [[Vikipedi:AWB|AWB]] ile,1777846053,CavlakBOT,true,false,true,"List(null, 145)","List(null, 5649277)",https://tr.wiktionary.org,tr.wiktionary.org,/w,trwiktionary,yeni sayfa AWB ile,2026-05-03T22:07:34.106136+00:00
/mediawiki/recentchange/1.0.0,"List(https://commons.wikimedia.org/wiki/Category:Media_lacking_a_description, 50b3fea4-0ae8-49d2-aa7c-12d95467f2db, faed9306-1782-4ed9-b1ae-18930194cbe4, 2026-05-03T22:07:34.076Z, commons.wikimedia.org,

Total records in Bronze: 10000


## 5. Camada Silver

A camada Silver é construída a partir da camada Bronze, mantendo a lógica Medallion do pipeline.

Nesta etapa, os eventos já persistidos em Parquet na Bronze são lidos novamente em modo streaming e submetidos a regras de validação, filtro e enriquecimento.

As regras aplicadas são:

- manter registros com `id` preenchido;
- manter registros com `type` preenchido;
- manter registros com `wiki` preenchido;
- manter registros com `title` preenchido;
- considerar apenas eventos dos tipos `edit`, `new`, `log` e `categorize`;
- converter o timestamp Unix para um campo temporal legível;
- calcular a diferença de tamanho da página quando essa informação estiver disponível.

Essa etapa demonstra a aplicação de validação e transformação dos dados dentro do pipeline de streaming.

In [0]:
from pyspark.sql.functions import (
    col,
    from_unixtime,
    to_timestamp,
    current_timestamp,
    when
)

bronze_stream_df = (
    spark.readStream
    .schema(wikimedia_schema)
    .option("maxFilesPerTrigger", 10)
    .parquet(BRONZE_OUTPUT_PATH)
)

print("Is bronze_stream_df streaming?", bronze_stream_df.isStreaming)

validated_stream_df = (
    bronze_stream_df
    .filter(col("id").isNotNull())
    .filter(col("type").isNotNull())
    .filter(col("wiki").isNotNull())
    .filter(col("title").isNotNull())
    .filter(col("type").isin("edit", "new", "log", "categorize"))
    .withColumn("event_timestamp", to_timestamp(from_unixtime(col("timestamp"))))
    .withColumn("ingestion_timestamp", to_timestamp(col("_ingestion_timestamp")))
    .withColumn(
        "length_delta",
        when(
            col("length.new").isNotNull() & col("length.old").isNotNull(),
            col("length.new") - col("length.old")
        ).otherwise(None)
    )
    .withColumn("processed_at", current_timestamp())
    .select(
        "id",
        "type",
        "wiki",
        "title",
        "user",
        "bot",
        "minor",
        "server_name",
        "event_timestamp",
        "ingestion_timestamp",
        "length_delta",
        "processed_at"
    )
)

# Confirma que o DataFrame validado continua sendo streaming.
print("Is validated_stream_df streaming?", validated_stream_df.isStreaming)

Is bronze_stream_df streaming? True
Is validated_stream_df streaming? True


In [0]:
silver_query = (
    validated_stream_df.writeStream
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH)
    .option("path", SILVER_OUTPUT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

silver_query.awaitTermination()

print("Silver streaming query completed.")
print("Query status:", silver_query.status)
print("Last progress:", silver_query.lastProgress)

Silver streaming query completed.
Query status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Last progress: {'id': '6e7f904a-66d7-4c77-9642-2359b2aea0a5', 'runId': '39db47ab-272d-41a0-985a-f03cbb048a70', 'name': None, 'timestamp': '2026-05-03T22:09:53.672Z', 'batchId': 0, 'batchDuration': 1353, 'durationMs': {'triggerExecution': 1353, 'queryPlanning': 62, 'collectSourceMetrics': 0, 'getBatch': 445, 'commitOffsets': 84, 'addBatch': 598, 'latestOffset': 74, 'walCommit': 167}, 'eventTime': {}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet]', 'startOffset': None, 'endOffset': '{"logOffset":0}', 'latestOffset': None, 'numInputRows': 9929, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 7338.507021433851, 'metrics': {}}], 'sink': {'description': 'FileSink[/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/val

In [0]:
silver_df = spark.read.parquet(SILVER_OUTPUT_PATH)

display(silver_df.limit(20))

print(f"Total records in Silver: {silver_df.count()}")

id,type,wiki,title,user,bot,minor,server_name,event_timestamp,ingestion_timestamp,length_delta,processed_at
3305921948,edit,commonswiki,File:Dohnaische Straße Pirna in color 119829381.jpg,SchlurcherBot,true,false,commons.wikimedia.org,2026-05-03T22:07:59.000Z,2026-05-03T22:07:59.538Z,2452,2026-05-03T22:09:53.747Z
3305921942,categorize,commonswiki,"Category:Maps showing Jan 25, 2022",ReneeEdits,false,null,commons.wikimedia.org,2026-05-03T22:07:58.000Z,2026-05-03T22:07:59.546Z,null,2026-05-03T22:09:53.747Z
80303791,categorize,cewiki,Категори:Unassessed geography articles,CheWikibot,true,null,ce.wikipedia.org,2026-05-03T22:07:58.000Z,2026-05-03T22:07:59.568Z,null,2026-05-03T22:09:53.747Z
3305921943,categorize,commonswiki,"Category:Jan 25, 2022 maps of Africa",ReneeEdits,false,null,commons.wikimedia.org,2026-05-03T22:07:58.000Z,2026-05-03T22:07:59.590Z,null,2026-05-03T22:09:53.747Z
3305921944,categorize,commonswiki,Category:SVG maps by Our World in Data,ReneeEdits,false,null,commons.wikimedia.org,2026-05-03T22:07:58.000Z,2026-05-03T22:07:59.612Z,null,2026-05-03T22:09:53.747Z
80303792,categorize,cewiki,Категори:Unknown-importance geography articles,CheWikibot,true,null,ce.wikipedia.org,2026-05-03T22:07:58.000Z,2026-05-03T22:07:59.634Z,null,2026-05-03T22:09:53.747Z
3305921947,edit,commonswiki,File:Mapillary (osmplus org) 2026-02-25 08H14M21S000 (956189414082421 at c4WOzsoTAYRf0DgXPjMZI2).jpg,Emijrpbot,true,false,commons.wikimedia.org,2026-05-03T22:07:58.000Z,2026-05-03T22:07:59.648Z,353,2026-05-03T22:09:53.747Z
80303793,categorize,cewiki,Категори:WikiProject Geography articles,CheWikibot,true,null,ce.wikipedia.org,2026-05-03T22:07:58.000Z,2026-05-03T22:07:59.656Z,null,2026-05-03T22:09:53.747Z
533843390,log,ruwiki,Участник:3.209.12.149,QBA-bot,true,null,ru.wikipedia.org,2026-05-03T22:07:59.000Z,2026-05-03T22:07:59.699Z,null,2026-05-03T22:09:53.747Z
146622508,edit,enwiktionary,jagirdar,~2026-25481-72,false,false,en.wiktionary.org,2026-05-03T22:07:58.000Z,2026-05-03T22:07:59.704Z,-3,2026-05-03T22:09:53.747Z


Total records in Silver: 9929


In [0]:
display(
    silver_df
    .groupBy("type")
    .count()
    .orderBy("count", ascending=False)
)

type,count
categorize,4773
edit,3158
log,1632
new,366


## 6. Camada Gold: Agregações Analíticas em Streaming

A camada Gold transforma os dados validados da Silver em indicadores de negócio. Utilizando agregações baseadas em janelas temporais de 5 minutos (Window Functions) e tolerância a atrasos (Watermarking), os eventos contínuos são agrupados em métricas acionáveis.

O `outputMode("append")` é utilizado em conjunto com o Watermarking: uma janela só é emitida após ser finalizada pelo watermark, evitando a reemissão de resultados parciais a cada micro-batch. Em execuções finitas com `trigger(availableNow=True)`, o watermark pode não avançar o suficiente para finalizar todas as janelas — comportamento esperado do Structured Streaming. Por isso, os resultados analíticos são exibidos abaixo a partir de leitura batch da camada Silver, aplicando as mesmas agregações e janelas.


In [0]:
from pyspark.sql.functions import window, col, count, sum as spark_sum, abs as spark_abs

silver_schema = spark.read.parquet(SILVER_OUTPUT_PATH).schema

silver_stream_df = (
    spark.readStream
    .schema(silver_schema)
    .option("maxFilesPerTrigger", 10)
    .parquet(SILVER_OUTPUT_PATH)
    .filter(col("event_timestamp").isNotNull())
    .withWatermark("event_timestamp", "10 minutes")
)

### Impacto de Conteúdo

Mede o volume real de caracteres alterados por Wiki. Indica a intensidade das mudanças na plataforma, permitindo identificar picos de atividade ou potenciais vandalismos em tempo real.

In [0]:
gold_impact_df = (
    silver_stream_df
    .groupBy(
        window(col("event_timestamp"), "5 minutes"),
        col("wiki")
    )
    .agg(
        count("*").alias("total_edits"),
        spark_sum(spark_abs(col("length_delta"))).alias("total_content_impact")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("total_edits"),
        col("total_content_impact")
    )
)

query_impact = (
    gold_impact_df.writeStream
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_GOLD_IMPACT_PATH)
    .option("path", f"{GOLD_AGG_OUTPUT_PATH}/window_impact")
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

### Engajamento de Usuários

Contabiliza o volume de ações realizadas, separando as contribuições de usuários humanos e de robôs (bots). Auxilia na compreensão do comportamento e engajamento da comunidade em cada janela de tempo.

In [0]:
gold_user_activity_df = (
    silver_stream_df
    .groupBy(
        window(col("event_timestamp"), "5 minutes"),
        col("user"),
        col("bot")
    )
    .agg(count("*").alias("total_events"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("user"),
        col("bot"),
        col("total_events")
    )
)

query_user = (
    gold_user_activity_df.writeStream
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_GOLD_USER_PATH)
    .option("path", f"{GOLD_AGG_OUTPUT_PATH}/user_activity")
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

### Perfil de Eventos

Classifica a quantidade de ações por tipo (edição, criação, categorização, etc.) em cada projeto. Mapeia a natureza das interações para entender se a Wiki está em fase de expansão de conteúdo ou apenas manutenção.

In [0]:
gold_event_types_df = (
    silver_stream_df
    .groupBy(
        window(col("event_timestamp"), "5 minutes"),
        col("wiki"),
        col("type")
    )
    .agg(count("*").alias("total_events"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("type"),
        col("total_events")
    )
)

query_event = (
    gold_event_types_df.writeStream
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_GOLD_EVENT_PATH)
    .option("path", f"{GOLD_AGG_OUTPUT_PATH}/event_types")
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

### Execução do Processamento

Aguarda a finalização da escrita de todos os relatórios processados nesta etapa de streaming.

In [0]:
query_impact.awaitTermination()
query_user.awaitTermination()
query_event.awaitTermination()

print("Tabelas analíticas da camada Gold geradas com sucesso.")

Tabelas analíticas da camada Gold geradas com sucesso.


In [0]:
# Status e progresso de cada query Gold
print("Gold Impact status:", query_impact.status)
print("Gold Impact last progress:", query_impact.lastProgress)
print("\nGold User Activity status:", query_user.status)
print("Gold User Activity last progress:", query_user.lastProgress)
print("\nGold Event Types status:", query_event.status)
print("Gold Event Types last progress:", query_event.lastProgress)

# Resultados analíticos: mesmas agregações e janelas das queries Gold,
# computadas via leitura batch da camada Silver.
silver_batch_df = (
    spark.read.parquet(SILVER_OUTPUT_PATH)
    .filter(col("event_timestamp").isNotNull())
)

print("\nImpacto de Conteúdo por Wiki:")
display(
    silver_batch_df
    .groupBy(window(col("event_timestamp"), "5 minutes"), col("wiki"))
    .agg(
        count("*").alias("total_edits"),
        spark_sum(spark_abs(col("length_delta"))).alias("total_content_impact")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("total_edits"),
        col("total_content_impact")
    )
    .orderBy("window_start", "wiki")
)

print("Engajamento de Usuários:")
display(
    silver_batch_df
    .groupBy(window(col("event_timestamp"), "5 minutes"), col("user"), col("bot"))
    .agg(count("*").alias("total_events"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("user"),
        col("bot"),
        col("total_events")
    )
    .orderBy("window_start")
)

print("Perfil de Eventos:")
display(
    silver_batch_df
    .groupBy(window(col("event_timestamp"), "5 minutes"), col("wiki"), col("type"))
    .agg(count("*").alias("total_events"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("type"),
        col("total_events")
    )
    .orderBy("window_start")
)

print(f"\nTotal records processed from Silver: {silver_batch_df.count()}")


Gold Impact status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Gold Impact last progress: {'id': 'adae5d01-86dc-4ba5-a33e-3ac7f5205882', 'runId': 'c896f42c-30e8-4b15-8c1d-34fef2fd9a99', 'name': None, 'timestamp': '2026-05-03T22:10:25.433Z', 'batchId': 1, 'batchDuration': 6966, 'durationMs': {'triggerExecution': 6966, 'queryPlanning': 38, 'collectSourceMetrics': 0, 'getBatch': 0, 'commitOffsets': 98, 'addBatch': 6739, 'latestOffset': 0, 'walCommit': 178}, 'eventTime': {'watermark': '2026-05-03T21:59:43.000Z'}, 'stateOperators': [{'operatorName': 'stateStoreSave', 'numRowsTotal': 83, 'numRowsUpdated': 0, 'allUpdatesTimeMs': 4, 'numRowsRemoved': 0, 'allRemovalsTimeMs': 470, 'commitTimeMs': 8889, 'memoryUsedBytes': 268566576, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 200, 'numStateStoreInstances': 200, 'customMetrics': {'rocksdbNumSnapshotsAutoRepaired': 0, 'rocksdbTotalBytesWrittenByFlush': 0, 'rocksdbGetLatency': 0, 'rocksdbNumInternalColFami

window_start,window_end,wiki,total_edits,total_content_impact
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,afwiki,2,16
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,anwiki,1,251
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,arwiki,35,2893
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,azwiki,144,2744
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,be_x_oldwiki,1,615
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,bewiki,2,70
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,bgwiki,2,12
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,bnwikibooks,2,274
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,bswiki,10,1
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,cawiki,15,2507


Engajamento de Usuários:


window_start,window_end,user,bot,total_events
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,Ssilvers,false,2
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,Jeffrey34555,false,1
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,Oklahoman79111728,false,1
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,Jonathan1,false,4
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,Zquid,false,1
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,Francisco Leandro,false,2
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,2pattywhack27,false,12
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,Paracel63,false,1
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,SongVĩ.Bot,true,1
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,BuzzerLone,false,5


Perfil de Eventos:


window_start,window_end,wiki,type,total_events
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,enwiktionary,edit,120
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,enwikisource,edit,6
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,wikidatawiki,new,193
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,bswiki,new,2
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,itwiki,new,1
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,urwiki,edit,31
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,elwiki,edit,5
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,dewiki,new,2
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,wikidatawiki,edit,802
2026-05-03T22:05:00.000Z,2026-05-03T22:10:00.000Z,eswiki,new,4



Total records processed from Silver: 9929


## 7. Evidências técnicas de streaming

In [0]:
# Consolida as principais evidências técnicas de que o pipeline foi executado com Spark Structured Streaming.
print("Streaming evidence")
print("------------------")

# Verifica se os DataFrames principais foram criados em modo streaming.
print("Raw DataFrame is streaming:", raw_stream_df.isStreaming)
print("Bronze DataFrame is streaming:", bronze_stream_df.isStreaming)
print("Validated Silver DataFrame is streaming:", validated_stream_df.isStreaming)
print("Gold Impact DataFrame is streaming:", gold_impact_df.isStreaming)
print("Gold User Activity DataFrame is streaming:", gold_user_activity_df.isStreaming)
print("Gold Event Types DataFrame is streaming:", gold_event_types_df.isStreaming)
print("Silver Stream DataFrame is streaming:", silver_stream_df.isStreaming)

# Exibe os caminhos de checkpoint usados por cada etapa streaming.
print("\nCheckpoint paths")
print("----------------")
print("Bronze checkpoint:", CHECKPOINT_BRONZE_PATH)
print("Silver checkpoint:", CHECKPOINT_SILVER_PATH)
print("Gold Impact checkpoint:", CHECKPOINT_GOLD_IMPACT_PATH)
print("Gold User Activity checkpoint:", CHECKPOINT_GOLD_USER_PATH)
print("Gold Event Types checkpoint:", CHECKPOINT_GOLD_EVENT_PATH)


Streaming evidence
------------------
Raw DataFrame is streaming: True
Bronze DataFrame is streaming: True
Validated Silver DataFrame is streaming: True
Gold Impact DataFrame is streaming: True
Gold User Activity DataFrame is streaming: True
Gold Event Types DataFrame is streaming: True
Silver Stream DataFrame is streaming: True

Checkpoint paths
----------------
Bronze checkpoint: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/bronze
Silver checkpoint: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/silver
Gold Impact checkpoint: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/gold_impact
Gold User Activity checkpoint: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/gold_user
Gold Event Types checkpoint: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/gold_event


## 8. Evidências dos outputs gerados

Os comandos abaixo evidenciam os arquivos criados em cada etapa do pipeline dentro do Unity Catalog Volume.

Essa verificação confirma a materialização das áreas Raw, Bronze, Silver e Gold.

In [0]:
# Exibe os arquivos brutos capturados na camada Raw.
print("Raw files")
display(dbutils.fs.ls(RAW_PATH))

Raw files


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events/wikimedia_events_20260503_220645.json,wikimedia_events_20260503_220645.json,14419578,1777846073000


In [0]:
print("Bronze output")
display(dbutils.fs.ls(BRONZE_OUTPUT_PATH))

Bronze output


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/_spark_metadata/,_spark_metadata/,0,1777846242650
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00000-d0ab6f55-17eb-41bf-af10-76d2844e48ef.c000.snappy.parquet,part-00000-d0ab6f55-17eb-41bf-af10-76d2844e48ef.c000.snappy.parquet,272551,1777846189000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00001-3d8bd690-48e7-414d-a05f-30e4fecc64b3.c000.snappy.parquet,part-00001-3d8bd690-48e7-414d-a05f-30e4fecc64b3.c000.snappy.parquet,269775,1777846189000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00002-3990fb6c-7e3f-45da-a8f7-38ba1bf8fe5a.c000.snappy.parquet,part-00002-3990fb6c-7e3f-45da-a8f7-38ba1bf8fe5a.c000.snappy.parquet,277906,1777846189000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00003-64ab5c68-6acd-4cda-aac9-420080cc0991.c000.snappy.parquet,part-00003-64ab5c68-6acd-4cda-aac9-420080cc0991.c000.snappy.parquet,275569,1777846189000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00004-abe9d118-caab-449a-9beb-88d3dfcc525a.c000.snappy.parquet,part-00004-abe9d118-caab-449a-9beb-88d3dfcc525a.c000.snappy.parquet,266884,1777846189000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00005-80a0335e-6a3b-4721-b0b3-e4fe28081bba.c000.snappy.parquet,part-00005-80a0335e-6a3b-4721-b0b3-e4fe28081bba.c000.snappy.parquet,271743,1777846189000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00006-9f51b1d8-fc48-4786-92ee-73585e19f76c.c000.snappy.parquet,part-00006-9f51b1d8-fc48-4786-92ee-73585e19f76c.c000.snappy.parquet,273801,1777846189000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00007-771c57d8-c5f3-4aef-8788-62b8d4736295.c000.snappy.parquet,part-00007-771c57d8-c5f3-4aef-8788-62b8d4736295.c000.snappy.parquet,138891,1777846189000


In [0]:
print("Silver output")
display(dbutils.fs.ls(SILVER_OUTPUT_PATH))

Silver output


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/_spark_metadata/,_spark_metadata/,0,1777846243557
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00000-babc5977-5faf-4d2b-a9f8-153e7ed93cc4.c000.snappy.parquet,part-00000-babc5977-5faf-4d2b-a9f8-153e7ed93cc4.c000.snappy.parquet,48270,1777846195000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00001-7dd66658-d0bc-4110-9e6f-7ac9780c05cc.c000.snappy.parquet,part-00001-7dd66658-d0bc-4110-9e6f-7ac9780c05cc.c000.snappy.parquet,49653,1777846195000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00002-01105205-7f04-49e5-82cd-216bb50a4a14.c000.snappy.parquet,part-00002-01105205-7f04-49e5-82cd-216bb50a4a14.c000.snappy.parquet,48063,1777846195000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00003-12419a2d-06b4-482c-bb9a-192c39ba60c4.c000.snappy.parquet,part-00003-12419a2d-06b4-482c-bb9a-192c39ba60c4.c000.snappy.parquet,47845,1777846195000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00004-ec995ca0-4fd7-4346-80e6-5cce001a7d16.c000.snappy.parquet,part-00004-ec995ca0-4fd7-4346-80e6-5cce001a7d16.c000.snappy.parquet,48917,1777846195000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00005-2525e7bb-3031-4c37-a60a-7d770d553710.c000.snappy.parquet,part-00005-2525e7bb-3031-4c37-a60a-7d770d553710.c000.snappy.parquet,47208,1777846195000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00006-45da2f19-5259-4a9a-92c7-50dd3d039442.c000.snappy.parquet,part-00006-45da2f19-5259-4a9a-92c7-50dd3d039442.c000.snappy.parquet,47236,1777846195000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00007-40b6955c-d9a4-443e-a00f-aa89fd066280.c000.snappy.parquet,part-00007-40b6955c-d9a4-443e-a00f-aa89fd066280.c000.snappy.parquet,25700,1777846195000


In [0]:
print("Gold aggregations output")
display(dbutils.fs.ls(GOLD_AGG_OUTPUT_PATH))

Gold aggregations output


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet/event_types/,event_types/,0,1777846244498
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet/user_activity/,user_activity/,0,1777846244498
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet/window_impact/,window_impact/,0,1777846244498
